In [7]:
import google.generativeai as genai
import configGemini

In [8]:
# Load your API key from config.py
api_key = configGemini.G_TOKENS
genai.configure(api_key=api_key)

In [9]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [4]:
def generate_text(prompt):
    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": 0.7,          # controls creativity (like OpenAI)
            "max_output_tokens": 256,     # like max_tokens
            "top_p": 0.9,                # nucleus sampling
            "top_k": 40                  # limits vocabulary diversity
        }
    )
    return response.text.strip()

In [5]:
prompt = "Write a 3-sentence story that begins with 'Once upon a time'."
generated_text = generate_text(prompt)
print(prompt, generated_text)

E0000 00:00:1765560837.620619 2383099 alts_credentials.cc:93] ALTS creds ignored. Not running on GCP and untrusted ALTS is not enabled.


Write a 3-sentence story that begins with 'Once upon a time'. Once upon a time, a lonely star wished for a friend to share its quiet nights. One evening, a passing comet, noticing its soft glow, paused its journey to listen to the star's ancient songs. From that moment on, the two celestial bodies orbited each other, illuminating the vast darkness with their shared, gentle light.


### Customizing Output

In [6]:
def generate_text(prompt, max_tokens, temperature):
    response = model.generate_content(
        prompt,
        generation_config={
            "temperature": temperature,         
            "max_output_tokens": max_tokens      
        }
    )
    return response.text.strip()

In [7]:
prompt = "Write a 3-sentence story that begins with 'Once upon a time'."
generated_text = generate_text(prompt, 200, 0)
print(prompt, generated_text)

Write a 3-sentence story that begins with 'Once upon a time'. Once upon a time, a curious squirrel named Nutmeg found a forgotten map tucked under an old oak tree. Following its faded lines, she


In [8]:
prompt = "Write a 3-sentence story that begins with 'Once upon a time'."
generated_text = generate_text(prompt, 200, 1)
print(prompt, generated_text)

Write a 3-sentence story that begins with 'Once upon a time'. Once upon a time, a curious


### Summarising Text

In [9]:
def text_summarizer(prompt):
    # Combine the multi-turn conversation into a single structured prompt
    messages = """
You will be provided with a block of text, and your task is to extract a list of keywords from it.

Example 1:
Text: A flying saucer seen by a guest house, a 7ft alien-like figure coming out of a hedge and a "cigar-shaped" UFO near a school yard.
Response: flying saucer, guest house, 7ft alien-like figure, hedge, cigar-shaped UFO, school yard, extraterrestrial encounters, UK, mass sightings, remote Welsh village, Broad Haven, Bermuda Triangle, mysterious craft sightings, strange beings, residents, single year, late seventies, Netflix documentary series, Steven Spielberg, production company, 1977, Cold War, Star Wars, Close Encounters of the Third Kind, science fiction blockbuster, box office.

Example 2:
Text: Each April, in the village of Maeliya in northwest Sri Lanka, Pinchal Weldurelage Siriwardene gathers his community under the shade of a large banyan tree...
Response: April, Maeliya, northwest Sri Lanka, Pinchal Weldurelage Siriwardene, banyan tree, wewa, reservoir, tank, Sinhala, rice paddies, 175-acres, 708,200 sq m, rainwater, agrarian committee, coconut milk, open hearth, blessings, prosperous harvest, deities, sluice gate, rice fields, irrigation canals, dry months, rains, lake-like water bodies, farmers, cultivate, Sinhala phrase, technology, village life, pagoda, temple.

Now extract keywords for the following text:
Text: {}
""".format(prompt)

    response = model.generate_content(
        messages,
        generation_config={
            "temperature": 0.5,          
            "max_output_tokens": 256     # equivalent to max_tokens
        }
    )

    # Safely extract text output
    if response.candidates and response.candidates[0].content.parts:
        return response.candidates[0].content.parts[0].text.strip()
    else:
        print("⚠️ No valid text output. Finish reason:", response.candidates[0].finish_reason)
        return ""


In [10]:
prompt = """Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more."""
print(prompt)

print("\nExtracted Keywords:\n")
print(text_summarizer(prompt))

Master Reef Guide Kirsty Whitman didn't need to tell me twice. Peering down through my snorkel mask in the direction of her pointed finger, I spotted a huge male manta ray trailing a female in perfect sync – an effort to impress a potential mate, exactly as Whitman had described during her animated presentation the previous evening. Having some knowledge of what was unfolding before my eyes on our snorkelling safari made the encounter even more magical as I kicked against the current to admire this intimate undersea ballet for a few precious seconds more.

Extracted Keywords:

Master Reef Guide, Kirsty Whitman, snorkel


### Poetic Chatbot

In [11]:
def poetic_chatbot(prompt):
    conversation = [
        {
            "role": "user",
            "parts": [{"text": "System: You are a poetic chatbot."}]
        },
        {
            "role": "user",
            "parts": [{"text": "When was Google founded?"}]
        },
        {
            "role": "model",
            "parts": [{"text": "In the late '90s, a spark did ignite, Google emerged, a radiant light. By Larry and Sergey, in '98, it was born, a search engine new, on the web it was sworn."}]
        },
        {
            "role": "user",
            "parts": [{"text": "Which country has the youngest president?"}]
        },
        {
            "role": "model",
            "parts": [{"text": "Ah, the pursuit of youth in politics, a theme we explore. In Austria, Sebastian Kurz did implore, at the age of 31, his journey did begin, leading with vigor, in a world filled with din."}]
        },
        {
            "role": "user",
            "parts": [{"text": prompt}]
        }
    ]

    response = model.generate_content(
        contents=conversation,
        generation_config={
            "temperature": 1,
            "max_output_tokens": 256,
        }
    )

    return response.text.strip()

In [12]:
prompt = "When was cheese first made?"
result=poetic_chatbot(prompt)
print(result)

Ah, a tale from the mists of


### Langchain

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
import google.generativeai as genai

In [2]:
url = "https://365datascience.com/upcoming-courses"

In [3]:
loader = WebBaseLoader(url)

In [4]:
raw_documents = loader.load()

In [5]:
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

In [12]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", api_key=api_key)

In [13]:
vectorstore = FAISS.from_documents(documents, embeddings)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0\n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0\n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0\n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 0', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/g

In [ ]:
memory = ConversationBufferMemory(memory_key = "chat_history", return_messages=True)

In [ ]:
qa = ConversationalRetrievalChain.from_llm(ChatOpenAI(openai_api_key=api_key, 
                                                  model="gpt-3.5-turbo", 
                                                  temperature=0), 
                                           vectorstore.as_retriever(), 
                                           memory=memory)

In [ ]:
query = "What is the next course to be uploaded on the 365DataScience platform?"

In [ ]:
result = qa({"question": query})

In [ ]:
result["answer"]